# SE1_data_access
This notebook will leverage the open source pygeohydro python notebook to retrieve hydrographic survey data catalogued in the USACE eHydro database. Usabel point cloud data is provided in .XYZ file format, as well as point shpaefiles within an ESRI .gdb file.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath('../src'))

from pathlib import Path
# import processing_help            # this is uncommented for SE1 submission as the rest of the repository would not be available
import warnings

# you can suppress some warnings if you wish. Some libraries further in the processing pipeline will benefit from this.
warnings.filterwarnings('ignore')

# Helpfer functions
- In final repository, these functions will be stored in the src/processing_help.py file for a cleaner document.
- For Water Digitilization, I have placed them below so the code may be executed without the entire repository.

In [ ]:
import os
from pygeohydro import EHydro
from concurrent.futures import ThreadPoolExecutor
from zipfile import ZipFile, BadZipFile

def extract_single_zip(zipf):
    """Extract a single zip file and remove it after extraction"""
    try:
        extract_dir = zipf.parent / zipf.stem
        extract_dir.mkdir(parents=True, exist_ok=True)
        with ZipFile(zipf, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        os.remove(zipf)
        return zipf.name, True
    except BadZipFile:
        os.remove(zipf)
        return zipf.name, False
    except Exception as e:
        return zipf.name, False

def retrieve_ehydro_data(data_dir, start_date, end_date, district_symbol, channel_area=None, surveyID = None, max_workers=8):
    """
    Function to retrieve eHydro survey data from the USACE database
    
    Parameters:
    -----------
    data_dir : Path
        Working directory for data storage
    start_date : str
        Start date in YYYY-MM-DD format
    end_date : str
        End date in YYYY-MM-DD format
    district_symbol : str
        USACE district code (e.g., 'CESWG')
    channel_area : str
        USACE NCF channel area named (e.g., 'CESWG_AN_01_BAY')
    surveyID : str
        USACE eHydro survey identifier (e.g., 'AN_01_BAY_20250523_CS')
    max_workers : int
        Number of parallel workers for zip extraction (default: 8)
    """
    where_clause = f"surveydatestart >= '{start_date}' AND surveydatestart <= '{end_date}' AND usacedistrictcode= '{district_symbol}'"
    data_dir = data_dir / district_symbol
    if surveyID:
        where_clause += f" AND surveyjobidpk= '{surveyID}'"

    if channel_area:
        where_clause += f" AND channelareaidfk= '{channel_area}'"
    
    data_dir.mkdir(parents=True, exist_ok=True)
    
    ehydro = EHydro(data_type="outlines", cache_dir=data_dir)
    topobathy = ehydro.bysql(where_clause)

    topobathy.to_parquet(data_dir / 'ehydro.parquet')
    print(f'eHydro survey data saved locally to {data_dir}')
    print(f'Survey metadata saved to {data_dir / "ehydro.parquet"}')

    # Parallel zip extraction for speed
    zip_files = list(data_dir.glob('*.ZIP')) + list(data_dir.glob('*.zip'))
    if zip_files:
        print(f'Extracting {len(zip_files)} zip files in parallel...')
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(extract_single_zip, zip_files))
        extracted = sum(1 for _, success in results if success)
        print(f'Successfully extracted {extracted}/{len(zip_files)} files')

    return topobathy

# Set up raw_data storage locally on your machine. This is where the hydrographic surveys will be saved from the eHydro ArcGIS REST server.

In [ ]:
# create a directory to story the data outside of the git repository

# the below example should just create 
DATA_DIR = Path(os.getcwd()).parent.parent / 'raw_data'

os.makedirs(DATA_DIR, exist_ok=True)

## Data description
An enterprise geospatial database used to process and disseminate hydrographic survey data for coastal and inland navigation channels. This data is used to inform navigational charts for waterways across the contiguous U.S., Alaska, Hawaii, and American territories. Data retrieved from the eHydro database will be in point cloud format stored as .XYZ, or within a point shapefile stored within an ESRI geodatabase. 

## How to run this notebook
1. Establish *DATA_DIR* as the directory where the raw data will be saved locally to your computer. This should not be pointing to the repository/working folder for this notebook. It is arbitrarily set to generate a new folder in your local Documents directory
2. Head over to the [eHydro ArcGIS Web Dashboard](https://www.arcgis.com/apps/dashboards/4b8f2ba307684cf597617bf1b6d2f85d) to click around and find hydrographic surveys you are interested in reprocessing. Pay attention to the dates of your search window, the string following **"District"**. Optionally the string following **"Survey ID:"** for each survey, and/or the strings in **'Channel ID'** list may be used to refine your search for surveys further.
3. Once you have identified the dates and the district for which you would want to download surveys from, enter them as strings assigned to the DISTRICT, START_DATE, and END_DATE variables. If you have a particular survey which you have identified and wish to download, enter that as a string assigned to the SURVEY_ID variable. If you have a particular channel area which you want to focus on, enter that string assigned to the CHANNEL_AREA variable.
4. Run the processing.retrieve_ehydro_data cell. This will download the identified eHydro surveys as zip files, extract them, remove the redundant zip files, and generate a parquet file storing the metadata for each downloaded survey which can be read in using pandas/geopandas. 

***YOU ARE DONE WITH RETRIEVEING DATA FOR AUTOBAG. WOOHOO!***

## Storage requirements
Data storage requirements is dependent on number of surveys examined, size of said surveys, as well as what data is provided. All of these vary by survey, so one certain measuremnt of storage requirement cannot be provided. Some users may be interested in one single hydrographic survey, while others may be interested in thousands. And, seeing as the way the data is collected over time (e.g., single-beam echosounders in the 1990s-2000s moving towards multi-beam echosounders in the 2010s-2020s) changes, the data sizes will also change drastically.

### Make sure you have sufficient storage available to download all data required!
- While these files are small, the eHydro database spans decades. If you wish to reprocess many surveys, file sizes can stack quickly and eat lots of your storage.

In [ ]:
# New York District
DISTRICT = 'CENAN'      # REQUIRED

# Kill Van Kull
CHANNEL_AREA = f'{DISTRICT}_NJ_05_AKN'      # OPTIONAL

# Kill Van Kull conditional survey from 08 JAN 2026
SURVEY_ID = 'NJ_05_AKN_20260108_CS_5638_30'      # OPTIONAL

# Define your search window here as strings
START_DATE = '2026-01-01'      # REQUIRED
END_DATE = '2026-03-08'      # REQUIRED

In [ ]:
# metadata = processing_help.retrieve_ehydro_data(DATA_DIR, START_DATE, END_DATE, DISTRICT, CHANNEL_AREA, SURVEY_ID, max_workers=8)       # you may change max_workers to increase or decrease download speed. I like having mine at 8

metadata = retrieve_ehydro_data(DATA_DIR, START_DATE, END_DATE, DISTRICT, CHANNEL_AREA, SURVEY_ID, max_workers=8)       # executes using the functions from within this notebook. The above is how it will actually be used in the repository/application

print(metadata.shape)
display(metadata.head())